In [ ]:
import pandas as pd
import pytz
from pvlib.location import Location
import datetime
  

coordinate_localita = {
    '7': {'latitude': 45.58, 'longitude': 7.81, 'altitude'=349},
    '13': {'latitude': 45.08, 'longitude': 8.27, 'altitude'=107},
    '19': {'latitude': 45.60, 'longitude': 8.42,'altitude'=241}, 
    '29': {'latitude': 45.02, 'longitude': 7.79,'altitude'=280},
    '36': {'latitude': 44.88, 'longitude': 7.34, 'altitude'=376},
    '45': {'latitude': 44.95, 'longitude': 8.86,'altitude'=85},
    '57': {'latitude': 45.13, 'longitude': 7.54, 'altitude'=365},
    '60': {'latitude': 44.57, 'longitude': 7.47, 'altitude'=476},
    '71': {'latitude': 44.84, 'longitude': 7.64, 'altitude'=241},
    '72': {'latitude': 44.69, 'longitude': 8.03, 'altitude'=172},
    '81': {'latitude': 45.38, 'longitude': 7.87, 'altitude'=270},
    '89': {'latitude': 45.07, 'longitude': 7.59, 'altitude'=293},
    '95': {'latitude': 45.37, 'longitude': 8.43, 'altitude'=126},
    '104': {'latitude': 45.08, 'longitude': 8.06, 'altitude'=},
    '107': {'latitude': 44.67, 'longitude': 8.38},
    '112': {'latitude': 44.73, 'longitude': 8.74},
    '113': {'latitude': 44.63, 'longitude': 8.58},
    '125': {'latitude': 44.67, 'longitude': 7.56},
    '128': {'latitude': 44.57, 'longitude': 7.76},
    '129': {'latitude': 44.37, 'longitude': 7.56},
    '130': {'latitude': 45.22, 'longitude': 8.11},
    '132': {'latitude': 44.72, 'longitude': 7.78},
    '134': {'latitude': 44.82, 'longitude': 8.13},
    '136': {'latitude': 44.46, 'longitude': 8.10},
    '143': {'latitude': 44.69, 'longitude': 8.07}
}
def remove_ambiguous_hours(df):
    """
    Rimuove l'ora ambigua durante il passaggio tra ora legale e ora solare (2:00 - 3:00).
    """
    # Localizza in UTC se le date non hanno un fuso orario
    if df['data'].dt.tz is None:
        df['data'] = df['data'].dt.tz_localize('UTC')
    
    # Converti in 'Europe/Rome'
    df['data'] = df['data'].dt.tz_convert('Europe/Rome')
    
    # Rimuove l'ora ambigua: 2:00 - 3:00 del giorno di cambio ora (ultima domenica di ottobre)
    df = df[~((df['data'].dt.month == 10) & 
              (df['data'].dt.day >= 25) &  # Consideriamo le ultime domeniche di ottobre
              (df['data'].dt.weekday == 6) &  # Domenica
              (df['data'].dt.hour >= 2) & (df['data'].dt.hour < 3))]
    
    return df

# Definire l'intervallo di date ridotto per il test
start_date = datetime.datetime(1996, 1, 1)
end_date = datetime.datetime(2023, 12, 31)
dates = pd.date_range(start=start_date, end=end_date, freq='T')  # Intervallo a livello di minuti

# Creazione del DataFrame finale per accumulare tutti i dati
final_df = pd.DataFrame()

# Loop su una sola località per il test
for localita, coord in coordinate_localita.items():
    latitude = coord['latitude']
    longitude = coord['longitude']
    
    print(f"Calcolo per la località {localita} (lat: {latitude}, lon: {longitude})")
    
    try:
        # Creare un oggetto Location di PVLib
        site = Location(latitude, longitude)
        print(f"  - Creata Location per {localita}")
        
        # Calcolare la clearsky radiation con il modello Ineichen minuto per minuto
        clearsky = site.get_clearsky(dates, model='ineichen')
        print(f"  - Calcolata clearsky radiation per {localita}")

        # Creare un DataFrame per la località con clearskyrad minuto per minuto
        df_localita_minuti = pd.DataFrame({
            'data': dates,
            'clearskyrad': clearsky['ghi'],
            'localita': localita
        })
        
        # Rimuovere l'ora ambigua
        df_localita_minuti = remove_ambiguous_hours(df_localita_minuti)
        
        # Aggiungere la colonna 'ora' prima di convertire la colonna 'data' in solo data
        df_localita_minuti['ora'] = df_localita_minuti['data'].dt.hour
        
        # Arrotonda le date all'inizio dell'ora
        df_localita_minuti['data'] = df_localita_minuti['data'].dt.floor('H')
        
        # Converti la colonna 'data' in formato yyyy-mm-dd
        df_localita_minuti['data'] = df_localita_minuti['data'].dt.date
        
        # Aggregare i dati su base oraria (media tra i minuti)
        df_localita_ora = df_localita_minuti.groupby(['localita', 'data', 'ora']).agg({
            'clearskyrad': 'mean'  # Media della clearsky radiation su base oraria
        }).reset_index()
        print(f"  - Aggregati i dati orari per {localita}")

        # Aggiungere i dati al DataFrame finale
        final_df = pd.concat([final_df, df_localita_ora])
        
    except Exception as e:
        print(f"Errore con la località {localita}: {e}")

# Resettare l'indice del DataFrame finale
final_df.reset_index(drop=True, inplace=True)

# Visualizza una parte del DataFrame finale
print(final_df.head())

# Salvare il DataFrame in un file CSV (opzionale)
final_df.to_csv('clearsky_radiation_data_oraria_test.csv', index=False)

print("Calcolo completato per il test. File salvato come 'clearsky_radiation_data_oraria_test.csv'.")


Calcolo per la località 7 (lat: 45.58, lon: 7.81)
  - Creata Location per 7
  - Calcolata clearsky radiation per 7
  - Aggregati i dati orari per 7
Calcolo per la località 19 (lat: 45.6, lon: 8.42)
  - Creata Location per 19
  - Calcolata clearsky radiation per 19
  - Aggregati i dati orari per 19
Calcolo per la località 29 (lat: 45.02, lon: 7.79)
  - Creata Location per 29
  - Calcolata clearsky radiation per 29
  - Aggregati i dati orari per 29
Calcolo per la località 36 (lat: 44.88, lon: 7.34)
  - Creata Location per 36
  - Calcolata clearsky radiation per 36
  - Aggregati i dati orari per 36
Calcolo per la località 45 (lat: 44.95, lon: 8.86)
  - Creata Location per 45
  - Calcolata clearsky radiation per 45
  - Aggregati i dati orari per 45
Calcolo per la località 57 (lat: 45.13, lon: 7.54)
  - Creata Location per 57
  - Calcolata clearsky radiation per 57
  - Aggregati i dati orari per 57
Calcolo per la località 60 (lat: 44.57, lon: 7.47)
  - Creata Location per 60
  - Calcolata c